## 1. Problem Definition

Boosting models are among the most powerful algorithms for structured/tabular fraud detection tasks.

Unlike bagging methods, boosting learns sequentially:

* each new model focuses on previous mistakes
* hard fraud cases receive more attention
* ensemble errors are gradually minimized

In fraud detection, boosting excels because fraud patterns are:

* highly non-linear
* behaviorally complex
* sparse and imbalanced
* interaction-heavy

**In this notebook, we will build:**

1. AdaBoost
2. Gradient Boosting
3. XGBoost (optional if installed)
4. LightGBM (optional if installed)

**We will evaluate:**

* Recall
* Precision
* F1-score
* ROC-AUC
* Feature Importance

## 2. Mathematical Intuition

**AdaBoost**

AdaBoost sequentially reweights difficult samples.

Misclassified fraud cases receive larger weights:

$$w_i \leftarrow w_i e^{\alpha}$$

**Final prediction:**

$$F(x) = \sum \alpha_m h_m(x)$$

Where:

* $h_m(x)$ = weak learner
* $\alpha_m$ = learner weight

---

**Gradient Boosting**

Gradient Boosting minimizes loss iteratively using gradient descent in function space.

General idea:

$$F_m(x)=F_{m-1}(x)+\gamma h_m(x)$$

Each new tree learns residual errors.

---

**XGBoost**

XGBoost improves Gradient Boosting using:

* regularization
* tree pruning
* parallelization
* second-order optimization

Widely used in:

* fraud detection
* credit scoring
* risk modeling
* Kaggle competitions

⸻

**LightGBM**

LightGBM introduces:

* histogram-based learning
* leaf-wise growth
* extremely fast training

Excellent for:

* large datasets
* production systems
* real-time risk scoring

## 3. Bias vs Variance

| Model | Bias | Variance |
| :--- | :--- | :--- |
| **AdaBoost** | Low to Moderate | High *(Without Regularization)* |
| **Gradient Boosting (GBM)** | Low | Moderate |
| **XGBoost** | Very Low | Low / Controlled |
| **LightGBM** | Very Low | Low / Controlled |

## 4. Setup & Imports

In [ ]:
# Setup
import sys
sys.path.append("..")

# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Boosting Models
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

# Project Modules
from src.data_loader import load_raw_data
from src.preprocess import preprocess_dataset
from src.config import RANDOM_STATE

# Styling
sns.set_style("whitegrid")

## 5. Load & Preprocess Data

In [ ]:
# Load dataset
df = load_raw_data()

# Preprocess dataset
X_train, X_test, y_train, y_test, preprocessor = (
    preprocess_dataset(df)
)

## 6. AdaBoost Classifier

**Initialize Model**

In [ ]:
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(
        max_depth=2,
        random_state=RANDOM_STATE
    ),
    n_estimators=200,
    learning_rate=0.05,
    random_state=RANDOM_STATE
)

**Train Model**

In [ ]:
ada_model.fit(X_train, y_train)

**Predictions**

In [ ]:
y_pred_ada = ada_model.predict(X_test)

y_pred_ada_proba = (
    ada_model.predict_proba(X_test)[:, 1]
)

**Evaluation**

In [ ]:
print(classification_report(y_test, y_pred_ada))

**ROC-AUC**

In [ ]:
roc_auc_ada = roc_auc_score(
    y_test,
    y_pred_ada_proba
)

print(f"AdaBoost ROC-AUC: {roc_auc_ada:.4f}")

**Confusion Matrix**

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_ada,
    cmap="Blues",
    ax=ax
)

plt.title("AdaBoost Confusion Matrix")
plt.show()

## 7. Gradient Boosting Classifier

**Initialize Model**

In [ ]:
gb_model = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    random_state=RANDOM_STATE
)

**Train Model**

In [ ]:
gb_model.fit(X_train, y_train)

**Predictions**

In [ ]:
y_pred_gb = gb_model.predict(X_test)

y_pred_gb_proba = (
    gb_model.predict_proba(X_test)[:, 1]
)

**Evaluation**

In [ ]:
print(classification_report(y_test, y_pred_gb))

**ROC-AUC**

In [ ]:
roc_auc_gb = roc_auc_score(
    y_test,
    y_pred_gb_proba
)

print(f"Gradient Boosting ROC-AUC: {roc_auc_gb:.4f}")

**ROC Curve**

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

RocCurveDisplay.from_predictions(
    y_test,
    y_pred_gb_proba,
    ax=ax
)

plt.title("Gradient Boosting ROC Curve")
plt.show()

## 8. XGBoost Classifier 

**Initialize Model**

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=7,
    eval_metric="logloss",
    random_state=RANDOM_STATE
)

**Train Model**

In [ ]:
xgb_model.fit(X_train, y_train)

**Predictions**

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)

y_pred_xgb_proba = (
    xgb_model.predict_proba(X_test)[:, 1]
)

**Evaluation**

In [ ]:
print(classification_report(y_test, y_pred_xgb))

**ROC-AUC**

In [ ]:
roc_auc_xgb = roc_auc_score(
    y_test,
    y_pred_xgb_proba
)

print(f"XGBoost ROC-AUC: {roc_auc_xgb:.4f}")

## 9. LightGBM Classifier 

**Initialize Model**

In [ ]:
lgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

**Train Model**

In [ ]:
lgbm_model.fit(X_train, y_train)

**Predictions**

In [ ]:
y_pred_lgbm = lgbm_model.predict(X_test)

y_pred_lgbm_proba = (
    lgbm_model.predict_proba(X_test)[:, 1]
)

**Evaluation**

In [ ]:
print(classification_report(y_test, y_pred_lgbm))

**ROC-AUC**

In [ ]:
roc_auc_lgbm = roc_auc_score(
    y_test,
    y_pred_lgbm_proba
)

print(f"LightGBM ROC-AUC: {roc_auc_lgbm:.4f}")

## 10. Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": [
        "AdaBoost",
        "Gradient Boosting",
        "XGBoost",
        "LightGBM"
    ],
    "ROC_AUC": [
        roc_auc_ada,
        roc_auc_gb,
        roc_auc_xgb,
        roc_auc_lgbm
    ]
})

results.sort_values(
    by="ROC_AUC",
    ascending=False
)

## 11. ROC Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

RocCurveDisplay.from_predictions(
    y_test,
    y_pred_ada_proba,
    name="AdaBoost",
    ax=ax
)

RocCurveDisplay.from_predictions(
    y_test,
    y_pred_gb_proba,
    name="Gradient Boosting",
    ax=ax
)

RocCurveDisplay.from_predictions(
    y_test,
    y_pred_xgb_proba,
    name="XGBoost",
    ax=ax
)

RocCurveDisplay.from_predictions(
    y_test,
    y_pred_lgbm_proba,
    name="LightGBM",
    ax=ax
)

plt.title("Boosting Models ROC Comparison")

plt.show()

## 12. Feature Importance Analysis

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
)

feature_importance.head(15)

**Visualization**

In [ ]:
plt.figure(figsize=(10, 8))

sns.barplot(
    data=feature_importance.head(15),
    x="importance",
    y="feature"
)

plt.title("Top XGBoost Features")

plt.show()

## 13. Interpretation

**observations:**

* AdaBoost outperform all prior models
* Boosting captures subtle fraud boundaries
* Engineered interaction features remain dominant
* Sequential learning improves difficult fraud detection

**dominant features:**

* anomaly_score
* device_anomaly_interaction
* composite_risk_score
* authentication_risk_score
* transaction_velocity_risk

---

## 14. Business Insights

**Boosting models are widely used in:**

* enterprise fraud detection
* credit risk systems
* anti-money laundering
* transaction scoring engines

**Advantages:**

* high predictive power
* excellent ranking ability
* strong fraud prioritization

**Operational benefits:**

* fewer missed fraud cases
* improved investigator efficiency
* better financial protection
* improved risk calibration

---

## 15. Limitations

**Boosting systems:**

* are computationally expensive
* may overfit noisy fraud patterns
* require hyperparameter tuning
* are harder to interpret

XGBoost and LightGBM especially require monitoring and calibration in production.

---

## 16. Conclusion

**In this notebook we:**

* implemented modern boosting architectures
* compared sequential ensemble learning methods
* evaluated fraud detection performance
* analyzed feature importance behavior
* benchmarked advanced boosting systems

Boosting methods are expected to become the strongest-performing fraud models in the project and will serve as leading candidates for production-grade fraud risk scoring systems.